<a href='https://colab.research.google.com/github/poggerssLL/Automatica---Grupo-2/blob/main/etapa-01-logica/09%20-%20Motor%20de%20Inferencia%20Forward%20e%20Backward%20Chaining.ipynb' target='_parent'><img src='https://colab.research.google.com/assets/colab-badge.svg' alt='Open In Colab'/></a>

# Aula 09 — Motor de Inferência Forward e Backward Chaining

Este notebook implementa o motor híbrido de inferência da máquina de envasamento de água em copos. A base de conhecimento da Aula 08 é processada de duas formas:

- **Forward chaining:** parte dos fatos observados e calcula o fechamento dedutivo até o ponto fixo.
- **Backward chaining:** parte de uma hipótese e constrói uma árvore de prova com regras, fatos, ausências e ciclos.

A implementação inclui resolução determinística de conflitos, trilha de auditoria e testes automatizados. O motor é uma camada didática de diagnóstico do SCADA; ele não substitui intertravamentos no controlador nem circuitos dedicados de segurança.

In [1]:
from dataclasses import dataclass
from typing import Any, Dict, Iterable, List, Optional, Set, Tuple


def formatar_tabela(dados: List[Dict[str, Any]]) -> str:
    '''Formata uma lista de dicionários como tabela ASCII, sem bibliotecas externas.'''
    if not dados:
        return 'Tabela vazia'

    colunas = list(dados[0].keys())
    larguras = {
        coluna: max(len(str(coluna)), *(len(str(linha.get(coluna, ''))) for linha in dados))
        for coluna in colunas
    }
    cabecalho = ' | '.join(f'{coluna:<{larguras[coluna]}}' for coluna in colunas)
    divisor = '-+-'.join('-' * larguras[coluna] for coluna in colunas)
    corpo = [
        ' | '.join(f'{str(linha.get(coluna, "")):<{larguras[coluna]}}' for coluna in colunas)
        for linha in dados
    ]
    return '\n'.join([cabecalho, divisor, *corpo])


print('[OK] Dependências carregadas: somente biblioteca padrão do Python.')

[OK] Dependências carregadas: somente biblioteca padrão do Python.


## 1. Modelo formal e estruturas do motor

Cada regra de produção possui a forma:

$$R_i: (A_{i,1} \land A_{i,2} \land \cdots \land A_{i,k}) \rightarrow C_i$$

No encadeamento para frente, todos os antecedentes devem pertencer à base de fatos ativa. No encadeamento para trás, o consequente é tratado como meta e seus antecedentes tornam-se submetas.

A prioridade de 1 a 10 abaixo é uma convenção local de arbitragem do projeto, e não uma classificação SIL.

In [2]:
@dataclass(frozen=True)
class RegraProducao:
    id_regra: str
    antecedentes: Tuple[str, ...]
    consequente: str
    diagnostico: str
    prioridade: int
    severidade: str
    pop: str

    def disparavel(self, fatos: Set[str]) -> bool:
        return set(self.antecedentes).issubset(fatos)


class BaseConhecimento:
    def __init__(self) -> None:
        self.regras: List[RegraProducao] = []

    def adicionar(self, regra: RegraProducao) -> None:
        if not regra.id_regra.strip():
            raise ValueError('O identificador da regra não pode ser vazio.')
        if not regra.antecedentes:
            raise ValueError(f'{regra.id_regra}: a regra deve possuir antecedentes.')
        if not regra.consequente.strip():
            raise ValueError(f'{regra.id_regra}: o consequente não pode ser vazio.')
        if not 1 <= regra.prioridade <= 10:
            raise ValueError(f'{regra.id_regra}: prioridade deve estar entre 1 e 10.')
        if any(existente.id_regra == regra.id_regra for existente in self.regras):
            raise ValueError(f'Identificador duplicado: {regra.id_regra}.')
        self.regras.append(regra)

    def regras_para(self, meta: str) -> List[RegraProducao]:
        candidatas = [regra for regra in self.regras if regra.consequente == meta]
        return sorted(candidatas, key=lambda r: (-r.prioridade, -len(r.antecedentes), r.id_regra))

    def validar(self) -> List[str]:
        problemas: List[str] = []
        for regra in self.regras:
            if regra.consequente in regra.antecedentes:
                problemas.append(f'{regra.id_regra}: autorreferência direta em {regra.consequente}.')
        return problemas


@dataclass
class ResultadoForward:
    fatos_iniciais: Set[str]
    fatos_finais: Set[str]
    trilha: List[Dict[str, Any]]

    @property
    def fatos_inferidos(self) -> Set[str]:
        return self.fatos_finais - self.fatos_iniciais


@dataclass
class NoProva:
    meta: str
    provado: bool
    origem: str
    regra: Optional[str] = None
    detalhe: str = ''
    filhos: Tuple['NoProva', ...] = ()


def arvore_para_linhas(raiz: NoProva) -> List[Dict[str, Any]]:
    linhas: List[Dict[str, Any]] = []

    def visitar(no: NoProva, nivel: int) -> None:
        linhas.append({
            'Nível': nivel,
            'Meta': f'{"  " * nivel}{no.meta}',
            'Provada': 'SIM' if no.provado else 'NÃO',
            'Origem': no.origem,
            'Regra': no.regra or '-',
            'Detalhe': no.detalhe,
        })
        for filho in no.filhos:
            visitar(filho, nivel + 1)

    visitar(raiz, 0)
    return linhas


class MotorInferencia:
    def __init__(self, base: BaseConhecimento) -> None:
        self.base = base

    def forward_chaining(self, fatos_iniciais: Iterable[str]) -> ResultadoForward:
        '''Calcula o ponto fixo da base e registra uma trilha cronológica.'''
        fatos_origem = set(fatos_iniciais)
        fatos = set(fatos_origem)
        regras_disparadas: Set[str] = set()
        trilha: List[Dict[str, Any]] = []

        while True:
            candidatas = [
                regra for regra in self.base.regras
                if regra.id_regra not in regras_disparadas
                and regra.disparavel(fatos)
                and regra.consequente not in fatos
            ]
            if not candidatas:
                break

            candidatas.sort(key=lambda r: (-r.prioridade, -len(r.antecedentes), r.id_regra))
            regra = candidatas[0]
            regras_disparadas.add(regra.id_regra)
            fatos.add(regra.consequente)
            trilha.append({
                'Passo': len(trilha) + 1,
                'Regra': regra.id_regra,
                'Prio': regra.prioridade,
                'Antecedentes': ' AND '.join(regra.antecedentes),
                'Fato inferido': regra.consequente,
                'Diagnóstico': regra.diagnostico,
                'POP': regra.pop,
            })

        return ResultadoForward(fatos_origem, fatos, trilha)

    def backward_chaining(self, meta: str, fatos_iniciais: Iterable[str]) -> NoProva:
        '''Tenta provar uma meta e devolve a árvore completa de justificativas.'''
        fatos = set(fatos_iniciais)
        memo: Dict[str, NoProva] = {}

        def provar(alvo: str, pilha: Tuple[str, ...]) -> NoProva:
            if alvo in fatos:
                return NoProva(alvo, True, 'FATO', detalhe='Fato presente na entrada do motor.')

            if alvo in pilha:
                ciclo = ' -> '.join((*pilha, alvo))
                return NoProva(alvo, False, 'CICLO', detalhe=f'Ciclo detectado: {ciclo}.')

            if alvo in memo:
                return memo[alvo]

            regras = self.base.regras_para(alvo)
            if not regras:
                no = NoProva(alvo, False, 'AUSENTE', detalhe='Sem fato de entrada e sem regra produtora.')
                memo[alvo] = no
                return no

            tentativas: List[NoProva] = []
            for regra in regras:
                filhos = tuple(provar(antecedente, (*pilha, alvo)) for antecedente in regra.antecedentes)
                if all(filho.provado for filho in filhos):
                    no = NoProva(
                        alvo,
                        True,
                        'REGRA',
                        regra=regra.id_regra,
                        detalhe=regra.diagnostico,
                        filhos=filhos,
                    )
                    memo[alvo] = no
                    return no

                tentativas.append(NoProva(
                    meta=f'Tentativa de {alvo}',
                    provado=False,
                    origem='REGRA_REJEITADA',
                    regra=regra.id_regra,
                    detalhe='Ao menos um antecedente não foi provado.',
                    filhos=filhos,
                ))

            no = NoProva(
                alvo,
                False,
                'NÃO_PROVADO',
                detalhe='Nenhuma regra candidata teve todos os antecedentes provados.',
                filhos=tuple(tentativas),
            )
            # Falhas com regras não são memorizadas: podem depender de um ciclo
            # provisório que outra rota de prova, em outro contexto, consiga resolver.
            return no

        return provar(meta, ())


print('[OK] Classes do motor híbrido definidas.')

[OK] Classes do motor híbrido definidas.


## 2. Base de conhecimento da envasadora

O catálogo abaixo mantém as sete regras introduzidas na Aula 08. Dessa forma, a Aula 09 acrescenta os algoritmos de inferência sem trocar o domínio diagnosticado.

In [3]:
def criar_base_envasadora() -> BaseConhecimento:
    base = BaseConhecimento()
    regras = [
        RegraProducao(
            'R-01', ('SOLICITA_GIRO_MESA', 'PRENSA_NAO_RECUADA'), 'TRIP_COLISAO_MESA',
            'Risco de colisão: mesa solicitada com prensa fora do recuo seguro.',
            10, 'EMERGÊNCIA',
            'POP-M01: bloquear M-001, comandar recuo seguro da prensa e alarmar.'
        ),
        RegraProducao(
            'R-02', ('SOLICITA_PRENSA', 'TEMPERATURA_ABAIXO_MIN'), 'BLOQUEIO_SELAGEM_FRIO',
            'Cabeçote térmico abaixo de 180 °C durante solicitação de prensagem.',
            8, 'ALTA',
            'POP-S04: inibir avanço de F e verificar HT-401 e TIT-401.'
        ),
        RegraProducao(
            'R-03', ('SOLICITA_GIRO_BRACO', 'VACUO_NAO_CONFIRMADO'), 'FALHA_CAPTURA_TAMPA',
            'Tampa não confirmada pela ventosa antes do giro do manipulador.',
            6, 'MÉDIA',
            'POP-P03: pausar XV-301 e inspecionar VAC-301 e o magazine de tampas.'
        ),
        RegraProducao(
            'R-04', ('CICLO_DISPENSA_CONCLUIDO', 'COPO_ESTACAO1_AUSENTE'), 'MAGAZINE_COPOS_VAZIO',
            'Ausência de copo após o ciclo de dispensação da Estação 1.',
            7, 'ALTA',
            'POP-D01: pausar a indexação, reabastecer o magazine e verificar ZS-102.'
        ),
        RegraProducao(
            'R-05', ('SOLICITA_DOSE_AGUA', 'BICO_FECHADO'), 'SOBREPRESSAO_DOSADOR',
            'Dosador solicitado com o bico indicado como fechado.',
            9, 'CRÍTICA',
            'POP-E02: abortar o avanço de C e levar o conjunto a estado seguro.'
        ),
        RegraProducao(
            'R-06', ('FIM_CURSO_AVANCO_ATIVO', 'FIM_CURSO_RECUO_ATIVO'), 'FALHA_INCOERENCIA_SENSOR',
            'Fins de curso antagônicos ativos simultaneamente no mesmo atuador.',
            8, 'ALTA',
            'POP-I01: inibir automático e solicitar inspeção elétrica dos sensores.'
        ),
        RegraProducao(
            'R-07', ('TRIP_COLISAO_MESA',), 'ALARME_GERAL_PARADA',
            'Parada geral derivada do trip de colisão da mesa.',
            10, 'EMERGÊNCIA',
            'POP-G00: desarmar comandos de movimento e acionar sinalização audiovisual.'
        ),
    ]
    for regra in regras:
        base.adicionar(regra)
    return base


base = criar_base_envasadora()
problemas = base.validar()
assert not problemas, problemas
motor = MotorInferencia(base)

catalogo = [
    {
        'ID': r.id_regra,
        'Antecedentes': ' AND '.join(r.antecedentes),
        'Consequente': r.consequente,
        'Prio': r.prioridade,
        'Severidade': r.severidade,
    }
    for r in base.regras
]
print(formatar_tabela(catalogo))
print('\n[OK] Base validada com 7 regras e identificadores únicos.')

ID   | Antecedentes                                       | Consequente              | Prio | Severidade
-----+----------------------------------------------------+--------------------------+------+-----------
R-01 | SOLICITA_GIRO_MESA AND PRENSA_NAO_RECUADA          | TRIP_COLISAO_MESA        | 10   | EMERGÊNCIA
R-02 | SOLICITA_PRENSA AND TEMPERATURA_ABAIXO_MIN         | BLOQUEIO_SELAGEM_FRIO    | 8    | ALTA      
R-03 | SOLICITA_GIRO_BRACO AND VACUO_NAO_CONFIRMADO       | FALHA_CAPTURA_TAMPA      | 6    | MÉDIA     
R-04 | CICLO_DISPENSA_CONCLUIDO AND COPO_ESTACAO1_AUSENTE | MAGAZINE_COPOS_VAZIO     | 7    | ALTA      
R-05 | SOLICITA_DOSE_AGUA AND BICO_FECHADO                | SOBREPRESSAO_DOSADOR     | 9    | CRÍTICA   
R-06 | FIM_CURSO_AVANCO_ATIVO AND FIM_CURSO_RECUO_ATIVO   | FALHA_INCOERENCIA_SENSOR | 8    | ALTA      
R-07 | TRIP_COLISAO_MESA                                  | ALARME_GERAL_PARADA      | 10   | EMERGÊNCIA

[OK] Base validada com 7 regras e identificadores únic

## 3. Ensaio 1 — Forward chaining e disparo em cascata

A solicitação de giro com a prensa fora do recuo seguro deve inferir o trip de colisão por R-01. Esse novo fato deve disparar R-07 e produzir o alarme geral.

In [4]:
fatos_colisao = {'SOLICITA_GIRO_MESA', 'PRENSA_NAO_RECUADA'}
resultado_colisao = motor.forward_chaining(fatos_colisao)

print(formatar_tabela(resultado_colisao.trilha))
print('\nFatos inferidos:', sorted(resultado_colisao.fatos_inferidos))

assert [passo['Regra'] for passo in resultado_colisao.trilha] == ['R-01', 'R-07']
assert resultado_colisao.fatos_inferidos == {'TRIP_COLISAO_MESA', 'ALARME_GERAL_PARADA'}
print('\n[OK] Cascata R-01 -> R-07 confirmada até o ponto fixo.')

Passo | Regra | Prio | Antecedentes                              | Fato inferido       | Diagnóstico                                                        | POP                                                                       
------+-------+------+-------------------------------------------+---------------------+--------------------------------------------------------------------+---------------------------------------------------------------------------
1     | R-01  | 10   | SOLICITA_GIRO_MESA AND PRENSA_NAO_RECUADA | TRIP_COLISAO_MESA   | Risco de colisão: mesa solicitada com prensa fora do recuo seguro. | POP-M01: bloquear M-001, comandar recuo seguro da prensa e alarmar.       
2     | R-07  | 10   | TRIP_COLISAO_MESA                         | ALARME_GERAL_PARADA | Parada geral derivada do trip de colisão da mesa.                  | POP-G00: desarmar comandos de movimento e acionar sinalização audiovisual.

Fatos inferidos: ['ALARME_GERAL_PARADA', 'TRIP_COLISAO_MESA']

[OK]

## 4. Ensaio 2 — Resolução de conflitos

Três falhas independentes são fornecidas simultaneamente. O motor deve selecionar primeiro R-05, de prioridade 9, e depois R-02 e R-06, ambas de prioridade 8. O identificador da regra resolve o empate final.

In [5]:
fatos_multiplos = {
    'SOLICITA_DOSE_AGUA', 'BICO_FECHADO',
    'SOLICITA_PRENSA', 'TEMPERATURA_ABAIXO_MIN',
    'FIM_CURSO_AVANCO_ATIVO', 'FIM_CURSO_RECUO_ATIVO',
}
resultado_multiplos = motor.forward_chaining(fatos_multiplos)
print(formatar_tabela(resultado_multiplos.trilha))

ordem = [passo['Regra'] for passo in resultado_multiplos.trilha]
assert ordem == ['R-05', 'R-02', 'R-06']
assert {'SOBREPRESSAO_DOSADOR', 'BLOQUEIO_SELAGEM_FRIO', 'FALHA_INCOERENCIA_SENSOR'}.issubset(
    resultado_multiplos.fatos_finais
)
print('\n[OK] Arbitragem determinística confirmada:', ' -> '.join(ordem))

Passo | Regra | Prio | Antecedentes                                     | Fato inferido            | Diagnóstico                                                         | POP                                                                   
------+-------+------+--------------------------------------------------+--------------------------+---------------------------------------------------------------------+-----------------------------------------------------------------------
1     | R-05  | 9    | SOLICITA_DOSE_AGUA AND BICO_FECHADO              | SOBREPRESSAO_DOSADOR     | Dosador solicitado com o bico indicado como fechado.                | POP-E02: abortar o avanço de C e levar o conjunto a estado seguro.    
2     | R-02  | 8    | SOLICITA_PRENSA AND TEMPERATURA_ABAIXO_MIN       | BLOQUEIO_SELAGEM_FRIO    | Cabeçote térmico abaixo de 180 °C durante solicitação de prensagem. | POP-S04: inibir avanço de F e verificar HT-401 e TIT-401.             
3     | R-06  | 8    | FIM_CURSO

## 5. Ensaios 3 e 4 — Backward chaining

Primeiro, o motor deve provar a meta ALARME_GERAL_PARADA a partir dos fatos de colisão. Depois, deve rejeitar a hipótese FALHA_CAPTURA_TAMPA quando apenas a solicitação de giro do braço é conhecida, registrando a falta de VACUO_NAO_CONFIRMADO.

A rejeição significa **meta não comprovada**, não normalidade garantida.

In [6]:
prova_alarme = motor.backward_chaining('ALARME_GERAL_PARADA', fatos_colisao)
print('ÁRVORE DE PROVA — META PROVADA')
print(formatar_tabela(arvore_para_linhas(prova_alarme)))
assert prova_alarme.provado
assert prova_alarme.regra == 'R-07'
assert prova_alarme.filhos[0].regra == 'R-01'

print('\nÁRVORE DE PROVA — META NÃO COMPROVADA')
prova_tampa = motor.backward_chaining('FALHA_CAPTURA_TAMPA', {'SOLICITA_GIRO_BRACO'})
print(formatar_tabela(arvore_para_linhas(prova_tampa)))
assert not prova_tampa.provado
linhas_tampa = arvore_para_linhas(prova_tampa)
assert any(linha['Meta'].strip() == 'VACUO_NAO_CONFIRMADO' and linha['Origem'] == 'AUSENTE' for linha in linhas_tampa)
print('\n[OK] Meta válida provada e antecedente ausente corretamente auditado.')

ÁRVORE DE PROVA — META PROVADA
Nível | Meta                   | Provada | Origem | Regra | Detalhe                                                           
------+------------------------+---------+--------+-------+-------------------------------------------------------------------
0     | ALARME_GERAL_PARADA    | SIM     | REGRA  | R-07  | Parada geral derivada do trip de colisão da mesa.                 
1     |   TRIP_COLISAO_MESA    | SIM     | REGRA  | R-01  | Risco de colisão: mesa solicitada com prensa fora do recuo seguro.
2     |     SOLICITA_GIRO_MESA | SIM     | FATO   | -     | Fato presente na entrada do motor.                                
2     |     PRENSA_NAO_RECUADA | SIM     | FATO   | -     | Fato presente na entrada do motor.                                

ÁRVORE DE PROVA — META NÃO COMPROVADA
Nível | Meta                               | Provada | Origem          | Regra | Detalhe                                                     
------+-------------------

## 6. Ensaios 5 e 6 — Consistência híbrida e detecção de ciclos

Para a mesma base inicial, uma conclusão obtida por forward chaining deve ser justificável pelo backward chaining. Por fim, uma base artificial com A → B e B → A verifica que o algoritmo termina mesmo sem fatos que sustentem o ciclo.

In [7]:
# Consistência entre os dois modos de inferência.
for meta in sorted(resultado_colisao.fatos_inferidos):
    prova = motor.backward_chaining(meta, fatos_colisao)
    assert prova.provado, f'A meta {meta} foi inferida para frente, mas não provada para trás.'
print('[OK] Toda conclusão do cenário de colisão possui prova regressiva.')

# Base cíclica mínima, deliberadamente sem fatos de apoio.
base_ciclica = BaseConhecimento()
base_ciclica.adicionar(RegraProducao('C-01', ('B',), 'A', 'A depende de B.', 1, 'TESTE', 'Nenhum'))
base_ciclica.adicionar(RegraProducao('C-02', ('A',), 'B', 'B depende de A.', 1, 'TESTE', 'Nenhum'))
motor_ciclico = MotorInferencia(base_ciclica)
prova_ciclica = motor_ciclico.backward_chaining('A', set())

print('\nÁRVORE DA BASE CÍCLICA')
print(formatar_tabela(arvore_para_linhas(prova_ciclica)))
assert not prova_ciclica.provado
assert any(linha['Origem'] == 'CICLO' for linha in arvore_para_linhas(prova_ciclica))
print('\n[OK] Ciclo fechado detectado e busca encerrada sem recursão infinita.')

# Um ciclo provisório não pode bloquear uma rota alternativa válida.
base_alternativa = BaseConhecimento()
base_alternativa.adicionar(RegraProducao('A-01', ('B',), 'A', 'Rota cíclica para A.', 2, 'TESTE', 'Nenhum'))
base_alternativa.adicionar(RegraProducao('A-02', ('C',), 'A', 'Rota válida para A.', 1, 'TESTE', 'Nenhum'))
base_alternativa.adicionar(RegraProducao('A-03', ('A',), 'B', 'B depende de A.', 1, 'TESTE', 'Nenhum'))
base_alternativa.adicionar(RegraProducao('A-04', ('A', 'B'), 'G', 'Meta final.', 1, 'TESTE', 'Nenhum'))
prova_alternativa = MotorInferencia(base_alternativa).backward_chaining('G', {'C'})
assert prova_alternativa.provado
print('[OK] Rota alternativa provou G mesmo após o encontro de um ciclo provisório.')

[OK] Toda conclusão do cenário de colisão possui prova regressiva.

ÁRVORE DA BASE CÍCLICA
Nível | Meta                 | Provada | Origem          | Regra | Detalhe                                                     
------+----------------------+---------+-----------------+-------+-------------------------------------------------------------
0     | A                    | NÃO     | NÃO_PROVADO     | -     | Nenhuma regra candidata teve todos os antecedentes provados.
1     |   Tentativa de A     | NÃO     | REGRA_REJEITADA | C-01  | Ao menos um antecedente não foi provado.                    
2     |     B                | NÃO     | NÃO_PROVADO     | -     | Nenhuma regra candidata teve todos os antecedentes provados.
3     |       Tentativa de B | NÃO     | REGRA_REJEITADA | C-02  | Ao menos um antecedente não foi provado.                    
4     |         A            | NÃO     | CICLO           | -     | Ciclo detectado: A -> B -> A.                               

[OK] Ciclo f

## 7. Resultado e limitações de engenharia

Os seis ensaios demonstram:

1. fechamento dedutivo por forward chaining;
2. disparo em cascata e resolução determinística de conflitos;
3. construção de árvore de prova por backward chaining;
4. rastreamento de fatos ausentes;
5. consistência entre inferência direta e prova regressiva;
6. proteção contra ciclos.

### Limitação de instrumentação

O fato BICO_FECHADO da regra R-05 exige confirmação positiva de fechamento. Como o catálogo atual informa apenas ZSC-203 para bico aberto, a negação desse sensor significa abertura não detectada, e não fechamento confirmado. Recomenda-se adicionar um fim de curso de bico fechado ou renomear o fato para ABERTURA_BICO_NAO_CONFIRMADA.

O motor trabalha com fatos monotônicos durante cada execução. Atualizações temporais, remoção de fatos e comandos reais continuam sob responsabilidade da FSM, do controlador e dos circuitos dedicados de segurança.

**Aula 09 validada: motor híbrido completo e auditável aplicado à máquina de envasamento.**